# Automated Domain Writing: Grid Pulsing

$_{Yongtao}$ $_{Liu,}$ $_{youngtaoliu@gmail.com}$

$_{February}$ $_{2026}$

### Import 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from aecroscopywave.wavebuilding import WaveGenerator
from aecroscopywave.interfaces import WaveVI, Cypher

### Initialize Experiment

##### Initialize an instance of PyCypher

In [ ]:
pycy = Cypher.PyCypher()

In [ ]:
# Test executing Igor Pro commands; you should see messages in Igor Pro's command window
pycy.igor.Execute("Print \"Igor Execute: Hello from Igor Pro!\"")
pycy.Execute("Print \"Execute: Hello from Igor Pro!\"")

##### Initialize an instance of PyWaveVI

In [ ]:
_path = r'C:\Users\PyScanner_FPGA_6124_01\PyScanner_FPGA_6124_01.exe'
pyvi = WaveVI.PyWaveVI(app_path = _path)

In [ ]:
# Set DAQ settings for PyWaveVI. It takes the default settings in LabView if no argument is provided
pyvi.set_IO_settings(IO_setting_dict ={"trigger_type": 0, "AO_amplifier": 2, "IO_timeout": 5, "AI_ch01": 1, "AI_ch02": 1, "AI_ch03": 1})

##### Set a wave

In [ ]:
t, square_wave = WaveGenerator.square_pulse(amplitude = -1, pulse_duration = [0.2, 5, 0.2])
plt.plot(t, square_wave)

##### Upload wave to VI

In [ ]:
pyvi.set_AO_waveforms(waveform=square_wave, zero_tail=False)

##### Check uploaded wave

In [ ]:
uploadedwave = pyvi.get_AO_waveforms()
plt.plot(uploadedwave)

##### Execute wave

In [ ]:
pyvi.set_IO_control(clear = True, upload = True, do_IO = True, fetch_result = False)

##### Move tip

In [ ]:
# Initialize tip movement
pycy.initialize_move_tip()

In [ ]:
# Move tip
pycy.move_tip(pixel_coordinates = [255, 255], transit_time = 0.5)

In [ ]:
# Engage tip or withdraw
pycy.engage()
# pycy.withdraw()

### Grid Pulsing

#### Set grid locations

In [ ]:
# All locations span across [start_point_x, end_point_x] in x-direction and [start_point_y, end_point_y] in y-direction.
# There are num_x rows and num_y columns in the locations array

start_point_x = 0   # Define location array parameters
end_point_x = 255
start_point_y = 0
end_point_y = 255
num_x = 20
num_y = 20

# Generate location array
pos_x = np.linspace(start_point_x, end_point_x, num_x, dtype = int)
pos_y = np.linspace(start_point_y, end_point_y, num_y, dtype = int)
pulse_pos = np.meshgrid(pos_x, pos_y)
pulse_pos_x = pulse_pos[0].reshape(-1)
pulse_pos_y = pulse_pos[1].reshape(-1)  # pulse_pos_x and pulse_pos_y are the coordinates of all locations

#### Set wave

In [ ]:
t, square_wave = WaveGenerator.square_pulse(amplitude = -3.5, pulse_duration = [0.2, 5, 0.2])
plt.plot(t, square_wave)

#### Upload wave to VI

In [ ]:
pyvi.set_AO_waveforms(waveform=square_wave, zero_tail=False)

#### Upload wave to DAQ

In [ ]:
pyvi.set_IO_control(clear = True, upload = True, do_IO = False, fetch_result = False)

#### Engage tip

In [ ]:
pycy.engage()

#### Start grid pulsing

In [ ]:
for i in range(len(pulse_pos_x)):
    #####################---Move tip to the pulse location---##################### 
    pycy.move_tip(pixel_coordinates = [pulse_pos_x[i], pulse_pos_y[i]], transit_time = 0.5)
    time.sleep(0.5)
    
    #####################---Apply pulse---##################### 
    pyvi.set_IO_control(clear = False, upload = False, do_IO = True, fetch_result = False)
    time.sleep(1)
    